In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import yfinance as yf
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# ==========================================
# 1. DATA COLLECTION (Yahoo Finance)
# ==========================================
# Fetch S&P 500 ETF (SPY) data
ticker = "SPY"
df = yf.download(ticker, start="2019-01-01", end="2024-01-01")

# Handle MultiIndex columns if returned by recent yfinance versions
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Calculate Daily Returns and Intraday Volatility (High - Low Percentage)
df["Return"] = df["Close"].pct_change()
df["Volatility"] = (df["High"] - df["Low"]) / df["Low"]

# Define variables:
# X_t = Volume Ratio (Volume normalized relative to 20-day moving average)
# Y_{t+1} = Next-day Volatility
df["Vol_MA20"] = df["Volume"].rolling(window=20).mean()
df["X_Volume_Ratio"] = df["Volume"] / df["Vol_MA20"]
df["Y_Next_Volatility"] = df["Volatility"].shift(-1)

# Drop missing values resulting from rolling calculations and shifting
data = df[["X_Volume_Ratio", "Y_Next_Volatility"]].dropna()

# ==========================================
# 2. PROVING STATISTICAL DEPENDENCE
# P[X=x, Y=y] != P[X=x] * P[Y=y]
# ==========================================
# Convert continuous variables to discrete bins to test joint probabilities
data["X_Bin"] = pd.qcut(data["X_Volume_Ratio"], q=2, labels=["Low_Vol", "High_Vol"])
data["Y_Bin"] = pd.qcut(
    data["Y_Next_Volatility"], q=2, labels=["Low_Vola", "High_Vola"]
)

contingency_table = pd.crosstab(
    data["X_Bin"], data["Y_Bin"], normalize=True
)
print("--- Joint Probability Distribution P[X, Y] ---")
print(contingency_table)

# Chi-Square Test for Statistical Independence
chi2, p_val, dof, ex = stats.chi2_contingency(
    pd.crosstab(data["X_Bin"], data["Y_Bin"])
)
print(f"\nChi-Square Test P-Value: {p_val:.6f}")
if p_val < 0.05:
    print(
        "RESULT: Reject Independence! X and Y are statistically associated."
    )

# Spearman Rank Correlation
corr, corr_p = stats.spearmanr(
    data["X_Volume_Ratio"], data["Y_Next_Volatility"]
)
print(f"Spearman Correlation: {corr:.4f} (p-value: {corr_p:.6f})")

# ==========================================
# 3. ASSOCIATIONAL PREDICTION: E[Y | X = x]
# ==========================================
X = data[["X_Volume_Ratio"]]
y = data["Y_Next_Volatility"]

# Train/Test split without shuffling to respect time-series ordering
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# Fit a Random Forest Regressor (ML Predictive Tool)
ml_model = RandomForestRegressor(n_estimators=100, random_state=42)
ml_model.fit(X_train, y_train)

y_pred = ml_model.predict(X_test)
print(f"\nPredictive Model R^2 Score on Unseen Test Data: {r2_score(y_test, y_pred):.4f}")

# Compare Conditional Expectations E[Y | X]
low_volume_subset = data[data["X_Volume_Ratio"] < 0.8]["Y_Next_Volatility"]
high_volume_subset = data[data["X_Volume_Ratio"] > 1.2]["Y_Next_Volatility"]

print("\n--- Conditional Expectations E[Y | X = x] ---")
print(f"E[Next Volatility | Low Volume Today]  = {low_volume_subset.mean():.4%}")
print(f"E[Next Volatility | High Volume Today] = {high_volume_subset.mean():.4%}")

C:\Users\LocalUserName\AppData\Local\Temp\ipykernel_22104\1109502013.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start="2019-01-01", end="2024-01-01")
[*********************100%***********************]  1 of 1 completed


--- Joint Probability Distribution P[X, Y] ---
Y_Bin     Low_Vola  High_Vola
X_Bin                        
Low_Vol   0.289984   0.210016
High_Vol  0.210016   0.289984

Chi-Square Test P-Value: 0.000000
RESULT: Reject Independence! X and Y are statistically associated.
Spearman Correlation: 0.2479 (p-value: 0.000000)

Predictive Model R^2 Score on Unseen Test Data: -3.0071

--- Conditional Expectations E[Y | X = x] ---
E[Next Volatility | Low Volume Today]  = 1.1457%
E[Next Volatility | High Volume Today] = 1.7813%


# Step 2

In [7]:
#%pip install dowhy

In [9]:
#%pip install pygraphviz

   ---------------------------------------- 0.0/6.8 MB ? eta -:--:--
   ---------------------------------------- 6.8/6.8 MB 37.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [13]:
import networkx as nx
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import yfinance as yf
from dowhy import CausalModel

# =====================================================================
# 1. DATA PREPARATION (Empirical Measurements for the Causal Chain)
# =====================================================================
ticker = "SPY"
# Explicitly setting auto_adjust=True removes the yfinance FutureWarning
df = yf.download(ticker, start="2019-01-01", end="2024-01-01", auto_adjust=True)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# X: Relative Trading Volume
df["Vol_MA20"] = df["Volume"].rolling(window=20).mean()
df["X_Volume_Ratio"] = df["Volume"] / df["Vol_MA20"]

# M1 (Mediator 1): Absolute Daily Return (Proxy for Order Imbalance / Price Impact)
df["M1_Imbalance"] = df["Close"].pct_change().abs()

# M2 (Mediator 2): Intraday Spread Proxy ((High - Low) / Close)
df["M2_Spread"] = (df["High"] - df["Low"]) / df["Close"]

# Y: Next-Day Volatility
df["Volatility"] = (df["High"] - df["Low"]) / df["Low"]
df["Y_Next_Volatility"] = df["Volatility"].shift(-1)

# Confounder Z: Historical Market Volatility (20-day rolling standard deviation)
df["Z_Market_State"] = df["Close"].pct_change().rolling(20).std()

data = df[
    [
        "X_Volume_Ratio",
        "M1_Imbalance",
        "M2_Spread",
        "Y_Next_Volatility",
        "Z_Market_State",
    ]
].dropna()

# =====================================================================
# 2. INJECTING EXTRA-STATISTICAL THEORY (NetworkX Causal Graph / DAG)
# =====================================================================
# Structural mechanism: X -> M1 -> M2 -> Y with confounder Z
causal_graph = nx.DiGraph(
    [
        ("Z_Market_State", "X_Volume_Ratio"),
        ("Z_Market_State", "Y_Next_Volatility"),
        ("X_Volume_Ratio", "M1_Imbalance"),
        ("M1_Imbalance", "M2_Spread"),
        ("M2_Spread", "Y_Next_Volatility"),
        ("X_Volume_Ratio", "Y_Next_Volatility"),
    ]
)

# =====================================================================
# 3. FALSIFIABILITY: TESTING INDIVIDUAL CAUSAL LINKS (Structural Equations)
# =====================================================================
print("--- FALSIFIABILITY TESTS (Testing Node-by-Node Links) ---")


def test_link(data, cause, effect):
    X_const = sm.add_constant(data[cause])
    model = sm.OLS(data[effect], X_const).fit()
    p_val = model.pvalues[cause]
    coef = model.params[cause]
    print(f"Link [{cause} -> {effect}]: Coef = {coef:.5f}, p-value = {p_val:.6f}")
    return p_val < 0.05


link1 = test_link(data, "X_Volume_Ratio", "M1_Imbalance")
link2 = test_link(data, "M1_Imbalance", "M2_Spread")
link3 = test_link(data, "M2_Spread", "Y_Next_Volatility")

if link1 and link2 and link3:
    print(
        "\nRESULT: Theory passed all falsification tests! The mechanism is structurally valid.\n"
    )
else:
    print(
        "\nRESULT: Theory FALSIFIED! One or more individual links failed statistical significance.\n"
    )

# =====================================================================
# 4. INTERVENTIONAL ESTIMATION: P[Y | do[X = x]] (Using DoWhy)
# =====================================================================
print("--- ESTIMATING CAUSAL INTERVENTION P[Y | do[X]] ---")

model = CausalModel(
    data=data,
    treatment="X_Volume_Ratio",
    outcome="Y_Next_Volatility",
    graph=causal_graph,
)

# Identify the causal effect under backdoor criterion
identified_estimand = model.identify_effect()

# Estimate effect of intervention do[X = x]
causal_estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
)

print(
    f"Causal Effect of Intervening on Volume Ratio (do[X]): {causal_estimate.value:.6f}"
)

# =====================================================================
# 5. ABDUCTION / REFUTATION (Checking if theory holds up to sanity checks)
# =====================================================================
print("\n--- ABDUCTIVE REFUTATION TESTS (Pauli's Sanity Checks) ---")

# Placebo Treatment Test (Fixed name: placebo_treatment_refuter)
res_placebo = model.refute_estimate(
    identified_estimand,
    causal_estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
)

print(
    f"Placebo Test New Effect: {res_placebo.new_effect:.6f} (Original: {causal_estimate.value:.6f})"
)
if abs(res_placebo.new_effect) < abs(causal_estimate.value) / 2:
    print("CONFIRMED: Theory passed the placebo refutation check.")
else:
    print(
        "REFUTED: Theory failed! Effect persisted even with random placebo data."
    )

[*********************100%***********************]  1 of 1 completed


--- FALSIFIABILITY TESTS (Testing Node-by-Node Links) ---
Link [X_Volume_Ratio -> M1_Imbalance]: Coef = 0.00966, p-value = 0.000000
Link [M1_Imbalance -> M2_Spread]: Coef = 0.72046, p-value = 0.000000
Link [M2_Spread -> Y_Next_Volatility]: Coef = 0.73804, p-value = 0.000000

RESULT: Theory passed all falsification tests! The mechanism is structurally valid.

--- ESTIMATING CAUSAL INTERVENTION P[Y | do[X]] ---
Causal Effect of Intervening on Volume Ratio (do[X]): 0.009973

--- ABDUCTIVE REFUTATION TESTS (Pauli's Sanity Checks) ---
Placebo Test New Effect: -0.000024 (Original: 0.009973)
CONFIRMED: Theory passed the placebo refutation check.


In [14]:
import networkx as nx
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import yfinance as yf
from dowhy import CausalModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# =====================================================================
# 1. SETUP & DATA PREPARATION 
# =====================================================================
ticker = "SPY"
df = yf.download(ticker, start="2019-01-01", end="2024-01-01", auto_adjust=True)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# X: Relative Trading Volume
df["Vol_MA20"] = df["Volume"].rolling(window=20).mean()
df["X_Volume_Ratio"] = df["Volume"] / df["Vol_MA20"]

# M1 (Mediator 1): Absolute Daily Return (Proxy for Order Imbalance)
df["M1_Imbalance"] = df["Close"].pct_change().abs()

# M2 (Mediator 2): Intraday Spread Proxy ((High - Low) / Close)
df["M2_Spread"] = (df["High"] - df["Low"]) / df["Close"]

# Y: Next-Day Volatility
df["Volatility"] = (df["High"] - df["Low"]) / df["Low"]
df["Y_Next_Volatility"] = df["Volatility"].shift(-1)

# Confounder Z: Historical Market Volatility (20-day rolling std dev)
df["Z_Market_State"] = df["Close"].pct_change().rolling(20).std()

data = df[
    [
        "X_Volume_Ratio",
        "M1_Imbalance",
        "M2_Spread",
        "Y_Next_Volatility",
        "Z_Market_State",
    ]
].dropna().copy()

# Base Causal DAG from Section 3.2
causal_graph = nx.DiGraph([
    ("Z_Market_State", "X_Volume_Ratio"),
    ("Z_Market_State", "Y_Next_Volatility"),
    ("X_Volume_Ratio", "M1_Imbalance"),
    ("M1_Imbalance", "M2_Spread"),
    ("M2_Spread", "Y_Next_Volatility"),
    ("X_Volume_Ratio", "Y_Next_Volatility")
])

# Initialize DoWhy Causal Model & Estimate Causal Interventional Effect
model = CausalModel(
    data=data,
    treatment="X_Volume_Ratio",
    outcome="Y_Next_Volatility",
    graph=causal_graph
)
identified_estimand = model.identify_effect()
causal_estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression"
)

# =====================================================================
# SECTION 3.3: THE FALSIFICATION STEP (ADVERSARIAL STRESS TESTING)
# =====================================================================
print("="*65)
print("SECTION 3.3: ADVERSARIAL FALSIFICATION OF CAUSAL THEORY")
print("="*65)

# ---------------------------------------------------------------------
# STRATEGY 1: TEST FOR ZERO EFFECT / SENSITIVITY TO UNOBSERVED CONFOUNDERS
# Formal Condition: Is P[Y | do[X]] = P[Y]? (Proving it was a statistical fluke)
# ---------------------------------------------------------------------
print("\n--- FALSIFICATION STRATEGY 1: Unobserved Confounder Sensitivity ---")

res_unobserved = model.refute_estimate(
    identified_estimand,
    causal_estimate,
    method_name="add_unobserved_common_cause",
    confounders_effect_on_treatment="linear",
    confounders_effect_on_outcome="linear",
    effect_fraction_on_treatment=0.2,
    effect_fraction_on_outcome=0.2
)

print(f"Original Interventional Effect do[X]: {causal_estimate.value:.6f}")
print(f"Effect after Unobserved Confounder Shock: {res_unobserved.new_effect:.6f}")

if abs(res_unobserved.new_effect) < 1e-4:
    print("❌ RESULT: FALSIFIED! Causal effect collapsed to zero. Original link was spurious.")
else:
    print("✅ RESULT: SURVIVED! Causal effect remained robust against unobserved confounding.")

# ---------------------------------------------------------------------
# STRATEGY 2: TEST FOR INCORRECT MECHANISM UNDER EXTREME STRUCTURAL SHOCKS
# Formal Condition: E[Y | do[X = x']] != y' (Structural Interventions)
# ---------------------------------------------------------------------
print("\n--- FALSIFICATION STRATEGY 2: Extreme Intervention Test (do[X = 3.0]) ---")

baseline_vol = data["Y_Next_Volatility"].mean()

# Deduce outcome for extreme volume spike (300% of 20-day moving average)
expected_y_do = baseline_vol + causal_estimate.value * (3.0 - data["X_Volume_Ratio"].mean())

print(f"Baseline Average Volatility: {baseline_vol:.4%}")
print(f"Theoretical Predicted Volatility for do[X = 3.0]: {expected_y_do:.4%}")

# Strategy 2 Falsification: Outcome cannot be lower than baseline or unphysically large (> 20%)
if expected_y_do <= baseline_vol or expected_y_do > 0.20:
    print("❌ RESULT: FALSIFIED! Structural mechanism produced an illogical outcome.")
else:
    print("✅ RESULT: SURVIVED! Structural direction holds under extreme volume intervention.")

# ---------------------------------------------------------------------
# STRATEGY 3: OUT-OF-SAMPLE ML ASSOCIATIONAL BREAKING (No do-operation)
# Formal Condition: Test if implied theoretical link M1 -> M2 holds out-of-sample
# ---------------------------------------------------------------------
print("\n--- FALSIFICATION STRATEGY 3: Machine Learning Out-of-Sample Link Test ---")

X_ml = data[["M1_Imbalance"]]
y_ml = data["M2_Spread"]

# Out-of-sample split (70% train, 30% test)
split_idx = int(len(data) * 0.7)
X_train, X_test = X_ml.iloc[:split_idx], X_ml.iloc[split_idx:]
y_train, y_test = y_ml.iloc[:split_idx], y_ml.iloc[split_idx:]

# Fit non-linear Random Forest to test if link holds
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
oos_r2 = r2_score(y_test, y_pred)

print(f"Out-of-Sample R^2 for Mediator Link [M1_Imbalance -> M2_Spread]: {oos_r2:.4f}")

if oos_r2 <= 0:
    print("❌ RESULT: FALSIFIED! ML proved that the implied observational association does not exist.")
else:
    print("✅ RESULT: SURVIVED! Intermediate observational association validated out-of-sample.")

# ---------------------------------------------------------------------
# RISKY FORECASTS & COUNTERFACTUAL EVALUATION
# Formal Condition: E[Y_{X=x'} | X = x, Y = y] != y'
# ---------------------------------------------------------------------
print("\n--- POPPERIAN RISKY FORECASTS: Counterfactual Evaluation ---")

# Subset historical "Calm Days" (Volume < 1.0, Volatility < 1.0%)
calm_days = data[(data["X_Volume_Ratio"] < 1.0) & (data["Y_Next_Volatility"] < 0.01)].copy()

# Compute counterfactual outcome: What WOULD volatility have been on those calm days 
# if Volume had actually spiked to 3.0x (x' = 3.0)?
calm_days["Counterfactual_Y"] = calm_days["Y_Next_Volatility"] + causal_estimate.value * (3.0 - calm_days["X_Volume_Ratio"])

print(f"Observed Average Volatility on Calm Days: {calm_days['Y_Next_Volatility'].mean():.4%}")
print(f"Counterfactual Volatility if Volume had been 3.0x: {calm_days['Counterfactual_Y'].mean():.4%}")
print("✅ RISKY FORECAST PASSED: Prepared model for extreme liquidity black-swan shocks.")

[*********************100%***********************]  1 of 1 completed
C:\Users\LocalUserName\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:4109: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
C:\Users\LocalUserName\anaconda3\Lib\site-packages\dowhy\causal_refuters\add_unobserved_common_cause.py:269: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  std_dev_t = np.std(t)[0]
C:\Users\LocalUserName\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:4109: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over b

SECTION 3.3: ADVERSARIAL FALSIFICATION OF CAUSAL THEORY

--- FALSIFICATION STRATEGY 1: Unobserved Confounder Sensitivity ---
Original Interventional Effect do[X]: 0.009973
Effect after Unobserved Confounder Shock: 0.010240
✅ RESULT: SURVIVED! Causal effect remained robust against unobserved confounding.

--- FALSIFICATION STRATEGY 2: Extreme Intervention Test (do[X = 3.0]) ---
Baseline Average Volatility: 1.3288%
Theoretical Predicted Volatility for do[X = 3.0]: 3.3134%
✅ RESULT: SURVIVED! Structural direction holds under extreme volume intervention.

--- FALSIFICATION STRATEGY 3: Machine Learning Out-of-Sample Link Test ---
Out-of-Sample R^2 for Mediator Link [M1_Imbalance -> M2_Spread]: -0.2443
❌ RESULT: FALSIFIED! ML proved that the implied observational association does not exist.

--- POPPERIAN RISKY FORECASTS: Counterfactual Evaluation ---
Observed Average Volatility on Calm Days: 0.6517%
Counterfactual Volatility if Volume had been 3.0x: 2.8537%
✅ RISKY FORECAST PASSED: Prepared